In [ ]:
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Optional

import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        get_linear_schedule_with_warmup,
        set_seed as hf_set_seed,
    )
except Exception:
    hf_set_seed = None

from sklearn.metrics import accuracy_score, f1_score, classification_report

#### Experiment Reproducibility

In [ ]:
# -----------------------------
# 0) Reproducibility
# -----------------------------
def seed_everything(
    seed: int = 42,
    *,
    use_cuda: bool = False,          # GPU 쓸 때만 True
    deterministic: bool = True,      # 재현성 우선(느려질 수 있음)
    enforce_determinism: bool = False,  # torch.use_deterministic_algorithms까지 강제
    set_env: bool = True,            # OS/라이브러리 레벨 환경변수도 세팅
) -> None:
    """
    실무용 시드/재현성 세팅 유틸.

    - CPU-only 기본값(use_cuda=False)
    - deterministic=True: cuDNN 결정론 옵션 + benchmark off
    - enforce_determinism=True: torch.use_deterministic_algorithms(True)로 비결정론 연산을 에러로 막음
      (실험/디버깅에 유용, production에서는 보통 False)
    """

    # -----------------------------
    # 1) Python / NumPy
    # -----------------------------
    random.seed(seed)               # 파이썬 random 난수 고정
    np.random.seed(seed)            # NumPy 난수 고정

    # -----------------------------
    # 2) PyTorch (CPU)
    # -----------------------------
    torch.manual_seed(seed)         # torch CPU 난수 고정

    # -----------------------------
    # 3) HuggingFace (선택)
    # -----------------------------
    if hf_set_seed is not None:
        hf_set_seed(seed)           # HF 내부(seed + 일부 라이브러리) 정리용

    # -----------------------------
    # 4) 환경변수(선택)
    # -----------------------------
    if set_env:
        # Python 해시 기반 연산(딕셔너리 순서 등) 안정화
        os.environ["PYTHONHASHSEED"] = str(seed)

        # CUDA에서 matmul 결정론 강화(AMP/TF32/커널 등에 따라 영향)
        # - 필요할 때만 켜는 게 일반적이라 use_cuda일 때만 설정 권장
        if use_cuda:
            # PyTorch 권장 설정 중 하나. 엄격/느슨 옵션이 있음.
            # ":4096:8"은 성능/호환성 밸런스, 더 엄격하게는 ":16:8" 등을 쓰기도 함.
            os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

    # -----------------------------
    # 5) GPU 설정(옵션)
    # -----------------------------
    if use_cuda and torch.cuda.is_available():
        torch.cuda.manual_seed(seed)        # 현재 GPU seed
        torch.cuda.manual_seed_all(seed)    # 멀티 GPU seed

        if deterministic:
            # cuDNN 결정론/벤치마크 설정 (CNN 계열에서 주로 영향)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

        # TF32는 속도 향상 대신 미세한 수치 차이를 만들 수 있어,
        # 재현성 최우선이면 끄는 경우가 많음(특히 Ampere+)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

    # -----------------------------
    # 6) 결정론 알고리즘 강제(옵션)
    # -----------------------------
    if enforce_determinism:
        # 비결정론 연산을 만나면 에러를 내서 "재현성 깨지는 지점"을 즉시 찾게 해줌
        # 경고만 받고 싶으면 warn_only=True를 고려
        torch.use_deterministic_algorithms(True)

        # PyTorch 일부 연산에서 경고/에러를 더 명확히 하려면 아래도 함께 쓰기도 함
        # (버전/환경에 따라 동작 차이 있음)
        # torch.set_deterministic_debug_mode("error")  # "default"|"warn"|"error"

# seed_everything(42)  # 기본이 CPU-only
# seed_everything(42, use_cuda=True, deterministic=True, enforce_determinism=False) # GPU 실험(재현성 우선)
# seed_everything(42, use_cuda=True, deterministic=True, enforce_determinism=True) # 재현성 깨지는 연산을 “잡아내고 싶을 때”



In [ ]:
# -----------------------------
# 1) Config
# -----------------------------
@dataclass  # Config라는 클래스를 dataclass로 정의 | 하이퍼파라미터/경로/플래그 관리용
class Config:
    model_name: str = "distilbert-base-uncased"  # 사용할 사전학습 모델 이름(허깅페이스 모델 ID)
    max_length: int = 256  # 토크나이저에서 max_length로 쓸 최대 토큰 길이 
    batch_size: int = 32
    lr: float = 2e-5
    weight_decay: float = 0.01  # AdamW의 weight decay 값(가중치 L2 정규화 유사), bias/LayerNorm에는 decay를 빼는 식으로 파라미터 그룹을 나누기도 함
    epochs: int = 3  # 전체 데이터셋을 몇 번 반복 학습할지
    warmup_ratio: float = 0.06  # 전체 스텝 대비 워밍업 비율. 예: 총 1000 step이면 60 step 동안 lr을 선형 증가시키고 이후 감소 스케줄.
    grad_clip: float = 1.0  # 그래디언트 클리핑 임계값, 폭주 방지

    # runtime
    use_cuda: bool = False
    fp16: bool = False  # GPU면 True 권장, mixed precision(fp16) 사용 여부
    num_workers: int = 2  # DataLoader(num_workers=...)에서 데이터 로딩 워커 프로세스 수 | CPU-only라도 워커를 늘리면 속도에 도움 될 수 있지만, 재현성/디버깅이 중요하면 0으로 두는 경우도 많음. 윈도우/WSL/노트북 환경에서 멀티프로세싱 이슈가 날 때가 있음.
    output_dir: str = "./ckpt_seqcls"  # 체크포인트/로그 저장 경로.
    use_class_weights: bool = False  # 클래스 불균형일 때 loss에 class weight를 적용할지 여부.

    @property
    def device(self) -> torch.device:
        """use_cuda 설정을 존중해서 device 결정(실무에서 강제 CPU 디버깅 가능)."""
        if self.use_cuda and torch.cuda.is_available():
            return torch.device("cuda")
        return torch.device("cpu")

    @property
    def use_fp16(self) -> bool:
        """fp16 플래그 + GPU일 때만 True (CPU에서 fp16로 터지는 사고 방지)."""
        return self.fp16 and self.device.type == "cuda"

CFG = Config(use_cuda=False, fp16=True)  # CPU면 use_fp16은 자동 False
device = CFG.device                      # torch.device('cpu')
# [GPU]
# CFG = Config(use_cuda=True, fp16=True)
# device = CFG.device                      # cuda 가능하면 cuda
# use_fp16 = CFG.use_fp16                  # cuda면 True


In [ ]:
# -----------------------------
# 2) Dataset
# -----------------------------
class TextClsDataset(Dataset):  # torch.utils.data.Dataset를 상속해서, DataLoader가 데이터를 꺼내갈 수 있는 형태로 만듭니다.
    def __init__(self, texts: List[str], labels: Optional[List[int]], tokenizer, max_length: int):
        self.texts = texts
        self.labels = labels
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):  # 데이터셋 길이(샘플 수) | DataLoader가 epoch 길이 계산/샘플링에 사용.
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, Any]:  # 인덱스 idx에 해당하는 1개 샘플을 반환. 반환 타입은 dict (HuggingFace 모델 입력 포맷과 호환되게).
        enc = self.tok(
            self.texts[idx],
            truncation=True,
            padding=False,
            max_length=self.max_length,
            return_tensors=None,
        )
        if self.labels is not None:
            enc["labels"] = int(self.labels[idx])
        return enc


def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    # padding을 batch 단위에서 처리 (실무에서 더 빠르고 유연)
    keys = batch[0].keys()
    has_labels = "labels" in keys

    input_ids = [item["input_ids"] for item in batch]
    attention_mask = [item["attention_mask"] for item in batch]
    # token_type_ids 없는 모델도 있음 (DistilBERT 등)
    token_type_ids = [item.get("token_type_ids") for item in batch]

    # pad
    max_len = max(len(x) for x in input_ids)
    def pad_1d(seq, pad_value=0):
        return seq + [pad_value] * (max_len - len(seq))

    input_ids = torch.tensor([pad_1d(x, 0) for x in input_ids], dtype=torch.long)
    attention_mask = torch.tensor([pad_1d(x, 0) for x in attention_mask], dtype=torch.long)

    batch_out = {"input_ids": input_ids, "attention_mask": attention_mask}

    if all(x is not None for x in token_type_ids):
        token_type_ids = torch.tensor([pad_1d(x, 0) for x in token_type_ids], dtype=torch.long)
        batch_out["token_type_ids"] = token_type_ids

    if has_labels:
        labels = torch.tensor([item["labels"] for item in batch], dtype=torch.long)
        batch_out["labels"] = labels

    return batch_out

NameError: name 'use_cuda' is not defined